# Chapter 2 — Shared memory: adjacent differences

Compute b[i] = a[i] - a[i-1] for i>0, with b[0]=0, in two ways:
directly from global memory and with a shared-memory buffer.

**Learning goals:** identify the central tile and halo, follow which thread loads
each value, understand the block barrier, and test whether this small amount of
reuse actually improves performance.

The colored figures are exact schematics of the indexing, not measurements.
GPU outputs and timings are generated only when you execute the CUDA program.
No measured outputs are included in this delivered notebook.

Figures are displayed directly below their cells. GPU results, hardware details,
and timing tables are printed in the notebook. No figure, CSV, Excel, or
result archive is exported. Only the CUDA source files and compiled executable
are written to the runtime, as required for compilation. Save the notebook
with its outputs if you want to retain the experiment.


## 1. Select and inspect a GPU runtime

In Colab select **Runtime → Change runtime type → GPU**. The notebook writes its
sources with `%%writefile` and compiles them with `nvcc`. No nvcc4jupyter, PyCUDA or
CuPy is required. Matplotlib is used only to draw the explanatory diagrams.


In [ ]:
!nvidia-smi
!nvcc --version
!mkdir -p code


## 2. See the direct reads

Consider an interior block of four threads starting at output index j=8.
The blue central tile contains a[8] through a[11]. The first output also needs
a[7], the orange **left halo**. A halo value is an ordinary input outside the
central tile, not a distinct memory technology or a fictitious boundary value.
No right halo is needed for this backward difference.

The direct kernel is `adjacent_direct`, defined in the next CUDA source cell.
It evaluates `a[i] - a[i-1]` directly, without a shared-memory buffer or
a block barrier. It issues two source-level input reads per valid interior output.
The four threads request eight inputs in total, but only five distinct values.
The arrows are dependencies, not a statement about exact DRAM transactions.


In [ ]:
"""Exact schematic diagrams, not measured GPU output. Reusable stencil palette."""
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Patch

BLUE='#236EA3'; LIGHT_BLUE='#DBECF7'
ORANGE='#B96513'; LIGHT_ORANGE='#FCE6CA'
GREEN='#207B58'; LIGHT_GREEN='#E0F0E8'
GRAY='#6E7882'; LIGHT_GRAY='#E9ECEF'; INK='#23313E'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,
                     'pdf.fonttype':42,'svg.fonttype':'none'})

def canvas(title,height=6.0):
    fig,ax=plt.subplots(figsize=(8.5,height))
    ax.set_xlim(-.75,5.0);ax.set_ylim(-.15,6.7);ax.axis('off')
    ax.text(-.65,6.47,title,fontsize=14,weight='bold',color=INK)
    return fig,ax

def cell(ax,x,y,label,kind='core',w=.84,h=.6,fs=13):
    edge,face={'core':(BLUE,LIGHT_BLUE),'halo':(ORANGE,LIGHT_ORANGE),
               'output':(GREEN,LIGHT_GREEN),'invalid':(GRAY,LIGHT_GRAY)}[kind]
    ax.add_patch(Rectangle((x-w/2,y-h/2),w,h,facecolor=face,edgecolor=edge,
                           linewidth=1.5,hatch='///' if kind=='invalid' else None,zorder=3))
    ax.text(x,y,label,ha='center',va='center',fontsize=fs,color=INK,zorder=4)

def arrow(ax,a,b,color=BLUE):
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',mutation_scale=11,
                                 lw=1.3,color=color,zorder=2))

def bracket(ax,left,right,y,text,color):
    ax.plot([left,left,right,right],[y-.1,y,y,y-.1],color=color,lw=1.3)
    ax.text((left+right)/2,y+.08,text,ha='center',va='bottom',color=color,fontsize=11)

def direct():
    fig,ax=canvas('A. Direct kernel: two input reads per thread',5.1)
    ax.text(-.65,5.83,'Interior block: B = 4, first output index j = 8',color=GRAY)
    for q in range(5):cell(ax,q,4.65,rf'$a_{{{7+q}}}$','halo' if q==0 else 'core')
    bracket(ax,-.42,.42,5.17,'Left halo',ORANGE)
    bracket(ax,.58,4.42,5.17,'Central tile',BLUE)
    for t in range(4):
        x=t+.5
        arrow(ax,(t,4.33),(x-.12,2.97),ORANGE if t==0 else BLUE)
        arrow(ax,(t+1,4.33),(x+.12,2.97),BLUE)
        cell(ax,x,2.48,rf'$t={t}$'+'\n'+rf'$b_{{{8+t}}}=a_{{{8+t}}}-a_{{{7+t}}}$',
             'output',w=.98,h=.92,fs=10)
    ax.text(2,1.48,'a8, a9 and a10 are requested by two neighboring threads.',ha='center',color=INK)
    ax.text(2,.86,'4 outputs  ×  2 global input reads  =  8 source-level reads',
            ha='center',weight='bold',color=INK)
    ax.text(2,.27,'Arrows show input dependencies, not DRAM transactions.',
            ha='center',fontsize=10,color=GRAY)
    return fig

def shared(partial=False):
    title='C. Last block: all threads reach the barrier' if partial else 'B. Shared kernel: load, synchronize, reuse'
    fig,ax=canvas(title,7.0)
    ax.text(-.65,5.99,'B = 4, j = 8, N = 10: only outputs b8 and b9 are valid' if partial else
            'Interior block: B = 4, first output index j = 8',color=GRAY)
    for q in range(5):
        invalid=partial and q>=3
        kind='invalid' if invalid else ('halo' if q==0 else 'core')
        cell(ax,q,5.23,'outside array' if invalid else rf'$a_{{{7+q}}}$',kind,fs=10 if invalid else 13)
        if not invalid:arrow(ax,(q,4.91),(q,3.87),ORANGE if q==0 else BLUE)
        ax.text(q,4.39,('t0: extra load' if q==0 else f't{q-1}: '+('write zero' if invalid else 'load')),
                ha='center',fontsize=10,color=GRAY if invalid else (ORANGE if q==0 else BLUE),
                bbox={'facecolor':'white','edgecolor':'none','pad':2},zorder=5)
        cell(ax,q,3.53,('0' if invalid else rf'$a_{{{7+q}}}$')+'\n'+f'tile[{q}]',kind,h=.74,fs=12)
    bracket(ax,-.42,.42,5.63,'Left halo',ORANGE)
    bracket(ax,.58,4.42,5.63,'Central tile',BLUE)
    ax.plot([-.45,4.45],[2.32,2.32],ls='--',color=INK,lw=1.4)
    ax.text(2,2.32,'  __syncthreads(): all 4 threads  ',ha='center',va='center',
            bbox={'facecolor':'white','edgecolor':'none','pad':3},fontfamily='monospace',fontsize=11,zorder=6)
    for t in range(4):
        valid=not partial or t<2
        x=t+.5
        if valid:
            arrow(ax,(t,3.12),(x-.12,1.73),ORANGE if t==0 else BLUE)
            arrow(ax,(t+1,3.12),(x+.12,1.73),BLUE)
        cell(ax,x,1.23,rf'$t={t}$'+'\n'+(rf'$b_{{{8+t}}}$' if valid else 'skip write'),
             'output' if valid else 'invalid',w=.98,h=.92,fs=11)
    ax.text(2,.42,'Thread zero loads a7 for output b8.' if partial else
            'B + 1 = 5 shared slots: five global reads supply four outputs.',ha='center',color=INK,fontsize=11)
    ax.text(2,.02,'Gray slots receive zero without an out-of-bounds input read.' if partial else
            'Orange: input outside the central tile, copied into this block\'s buffer.',
            ha='center',color=GRAY,fontsize=10)
    return fig

def dependencies(B=4,left=2,right=2,j=8):
    """Reusable stencil view: choose B, left/right reach, and first output j."""
    count=B+left+right
    fig,ax=plt.subplots(figsize=(8.5,5.3))
    ax.set_xlim(-2.0,count-.35);ax.set_ylim(-1.45,B+2.0);ax.axis('off')
    ax.text(-1.85,B+1.57,'D. A wider stencil reuses more inputs',fontsize=14,weight='bold',color=INK)
    ax.text(-1.85,B+.95,f'B = {B}; left halo = {left}; right halo = {right}. '
            'Each row marks one output.',fontsize=11,color=GRAY)
    for q in range(count):
        kind='halo' if q<left or q>=left+B else 'core'
        cell(ax,q,B+.2,rf'$a_{{{j-left+q}}}$',kind,w=.84,h=.48,fs=12)
    for t in range(B):
        y=B-1-t
        ax.text(-.75,y,rf'$t={t}:\ b_{{{j+t}}}$',ha='right',va='center',color=GREEN,fontsize=12)
        for q in range(count):
            if t<=q<=t+left+right:
                kind='halo' if q<left or q>=left+B else 'core'
                cell(ax,q,y,'read',kind,w=.84,h=.58,fs=10)
            else:
                ax.add_patch(Rectangle((q-.42,y-.29),.84,.58,facecolor='#FAFAFA',edgecolor='#DFE4E8'))
    reads=B*(left+right+1)
    ax.text((count-1)/2,-.73,f'Direct: {reads} global input reads.  Shared: {count} global loads, then reuse.',
            ha='center',weight='bold',fontsize=11,color=INK)
    ax.text((count-1)/2,-1.2,'Blue: central tile     Orange: halo     Green: outputs / threads',ha='center',fontsize=10,color=GRAY)
    return fig



In [ ]:
from IPython.display import display
fig = direct()
display(fig)
plt.close(fig)


## 3. Write the two CUDA kernels

`adjacent_direct` is the baseline: each thread reads the two inputs it needs.
`adjacent_shared` instead cooperates with the other threads in its block.

The second kernel loads a central value into `tile[t+1]` for each thread t.
Thread zero performs one additional load: the halo goes into `tile[0]`.
The array named `tile` includes both the blue central tile and the orange halo.
Its size is therefore B+1 floats, specified at launch.

`__syncthreads()` comes after the loads and before the threads reuse each other's
values. Each block owns a separate buffer. A block's halo is read from global
memory, never from the preceding block's shared-memory buffer.


In [ ]:
%%writefile code/chapter02_adjacent_kernels.cuh
#ifndef BOOK_ADJACENT_KERNELS_CUH
#define BOOK_ADJACENT_KERNELS_CUH
extern "C" __global__
void adjacent_direct(const float *a, float *b, int n) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) b[i] = (i == 0) ? 0.0f : a[i] - a[i - 1];
}

extern "C" __global__
void adjacent_shared(const float *a, float *b, int n) {
    extern __shared__ float tile[];
    const int t = threadIdx.x;
    const int i = blockIdx.x * blockDim.x + t;
    tile[t + 1] = (i < n) ? a[i] : 0.0f;
    if (t == 0) {
        tile[0] = (i > 0 && i < n) ? a[i - 1] : 0.0f;
    }
    __syncthreads();
    if (i < n) {
        b[i] = (i == 0) ? 0.0f : tile[t + 1] - tile[t];
    }
}
#endif


In [ ]:
fig = shared()
display(fig)
plt.close(fig)


## 4. Follow the partial last block

For N=10 and B=4, the last block starts at i=8. Threads zero and one produce b[8]
and b[9]. Threads two and three have no valid output. They set their local slots
to zero **without reading a[10] or a[11]**, still reach the barrier, then skip the
output write. Thread zero loads a[7] as the halo.

At the other end of the array, the first block sets `tile[0]=0` without reading
a[-1]; b[0]=0 is imposed explicitly. That value of b[0] is the prescribed rule,
not a subtraction from a fictitious zero-valued sample.


In [ ]:
fig = shared(partial=True)
display(fig)
plt.close(fig)


## 5. Timing support and host program

The support files reuse the author's adapted `TimingGPU` interface. Events are
checked, reused and released. The original timer implementation need not be
revisited to understand this example. `GetCounter()` waits and returns milliseconds.

The host launches the shared kernel with:

```cpp
const size_t shared_bytes = (threads + 1) * sizeof(float);
adjacent_shared<<<blocks, threads, shared_bytes>>>(d_a, d_b, n);
```

The third launch argument reserves the buffer separately for each block.
The program provides three modes: print a small demonstration, check boundary
cases, and benchmark both methods. Each output buffer is filled with NaNs before
validation so that missing output writes are detected.


In [ ]:
%%writefile code/cuda_check.cuh
#ifndef BOOK_CUDA_CHECK_CUH
#define BOOK_CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    const cudaError_t status_ = (call); \
    if (status_ != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status_)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


In [ ]:
%%writefile code/TimingGPU.cuh
#ifndef BOOK_TIMING_GPU_CUH
#define BOOK_TIMING_GPU_CUH
// Adapted from the author's Timing.zip for repeated notebook experiments.
// Same public timing methods; errors checked and resources released.
struct PrivateTimingGPU;
class TimingGPU {
    PrivateTimingGPU *privateTimingGPU;
    void Start(unsigned int flags);
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    void StartCounterFlags();
    float GetCounter(); // milliseconds; waits for the stop event
};
#endif


In [ ]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"

struct PrivateTimingGPU {
    cudaEvent_t start, stop;
    unsigned int flags;
    bool started;
};

TimingGPU::TimingGPU() : privateTimingGPU(new PrivateTimingGPU{}) {
    CUDA_CHECK(cudaEventCreate(&privateTimingGPU->start));
    CUDA_CHECK(cudaEventCreate(&privateTimingGPU->stop));
}

TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(privateTimingGPU->start));
    CUDA_CHECK(cudaEventDestroy(privateTimingGPU->stop));
    delete privateTimingGPU;
}

void TimingGPU::Start(unsigned int flags) {
    if (flags != privateTimingGPU->flags) {
        CUDA_CHECK(cudaEventDestroy(privateTimingGPU->start));
        CUDA_CHECK(cudaEventDestroy(privateTimingGPU->stop));
        CUDA_CHECK(cudaEventCreateWithFlags(&privateTimingGPU->start, flags));
        CUDA_CHECK(cudaEventCreateWithFlags(&privateTimingGPU->stop, flags));
        privateTimingGPU->flags = flags;
    }
    CUDA_CHECK(cudaEventRecord(privateTimingGPU->start, 0));
    privateTimingGPU->started = true;
}

void TimingGPU::StartCounter() { Start(cudaEventDefault); }
void TimingGPU::StartCounterFlags() { Start(cudaEventBlockingSync); }

float TimingGPU::GetCounter() {
    if (!privateTimingGPU->started) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(privateTimingGPU->stop, 0));
    CUDA_CHECK(cudaEventSynchronize(privateTimingGPU->stop));
    float elapsed_ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&elapsed_ms, privateTimingGPU->start,
                                  privateTimingGPU->stop));
    return elapsed_ms;
}


In [ ]:
%%writefile code/chapter02_adjacent_shared.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include "chapter02_adjacent_kernels.cuh"
#include <algorithm>
#include <array>
#include <cmath>
#include <cstring>
#include <random>
#include <vector>

const char *names[]={"direct","shared"};
struct Data {
    int n;
    size_t bytes;
    std::vector<float> a;
    float *da=nullptr,*db=nullptr;
    explicit Data(int length) : n(length), bytes(size_t(n)*sizeof(float)), a(n) {
        for(int i=0;i<n;++i) {int k=i%17; a[i]=0.25f*float(k*k);}
        CUDA_CHECK(cudaMalloc((void**)&da,bytes));
        CUDA_CHECK(cudaMalloc((void**)&db,bytes));
        CUDA_CHECK(cudaMemcpy(da,a.data(),bytes,cudaMemcpyHostToDevice));
    }
    ~Data() {CUDA_CHECK(cudaFree(da));CUDA_CHECK(cudaFree(db));}
    void launch(int method,int threads) {
        const int blocks=(n+threads-1)/threads;
        if(method==0) adjacent_direct<<<blocks,threads>>>(da,db,n);
        else {
            const size_t shared_bytes=size_t(threads+1)*sizeof(float);
            adjacent_shared<<<blocks,threads,shared_bytes>>>(da,db,n);
        }
    }
    std::vector<float> validate(int method,int threads) {
        CUDA_CHECK(cudaMemset(db,0xff,bytes));
        launch(method,threads);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaDeviceSynchronize());
        std::vector<float> out(n);
        CUDA_CHECK(cudaMemcpy(out.data(),db,bytes,cudaMemcpyDeviceToHost));
        for(int i=0;i<n;++i) {
            float ref=i==0?0.f:a[i]-a[i-1];
            if(!std::isfinite(out[i]) || std::fabs(out[i]-ref)>1.e-6f*(1.f+std::fabs(ref))) {
                std::fprintf(stderr,"Validation failed: %s, N=%d, B=%d, i=%d\n",names[method],n,threads,i);
                std::exit(EXIT_FAILURE);
            }
        }
        return out;
    }
};

double batch(Data& data,int method,int threads,int repeats,TimingGPU& timer) {
    timer.StartCounter();
    for(int k=0;k<repeats;++k) data.launch(method,threads);
    CUDA_CHECK(cudaGetLastError());
    return timer.GetCounter()/repeats;
}

double quantile(std::vector<double> values,double q) {
    std::sort(values.begin(),values.end());
    double pos=q*(values.size()-1);
    size_t lo=size_t(pos),hi=std::min(lo+1,values.size()-1);
    return values[lo]+(values[hi]-values[lo])*(pos-lo);
}

void benchmark(int n,int threads,std::mt19937& rng) {
    Data data(n);
    TimingGPU timer;
    int repeats[2];
    std::vector<double> times[2];
    for(int method=0;method<2;++method) {
        data.validate(method,threads);
        for(int k=0;k<2;++k) data.launch(method,threads);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaDeviceSynchronize());
        double pilot=batch(data,method,threads,1,timer);
        repeats[method]=int(std::min(200.0,std::max(1.0,std::ceil(5.0/std::max(pilot,0.001)))));
    }
    for(int trial=0;trial<7;++trial) {
        std::array<int,2> order{{0,1}};
        std::shuffle(order.begin(),order.end(),rng);
        for(int method:order) {
            data.launch(method,threads);
            CUDA_CHECK(cudaGetLastError());
            CUDA_CHECK(cudaDeviceSynchronize());
            double ms=batch(data,method,threads,repeats[method],timer);
            if(!(ms>0) || !std::isfinite(ms)) {
                std::fprintf(stderr,"Invalid event interval.\n");std::exit(EXIT_FAILURE);
            }
            times[method].push_back(ms);
        }
    }
    double med[2];
    std::printf("\nN=%d, B=%d (all outputs validated)\n",n,threads);
    std::printf("method  median(us)  Q25(us)  Q75(us)\n");
    for(int method=0;method<2;++method) {
        med[method]=quantile(times[method],.5);
        double q25=quantile(times[method],.25),q75=quantile(times[method],.75);
        std::printf("%-6s %10.3f %8.3f %8.3f\n",names[method],1000*med[method],1000*q25,1000*q75);
    }
    std::printf("T_direct / T_shared = %.3f (>1 means shared is faster)\n",med[0]/med[1]);
}

int main(int argc,char** argv) {
    if(argc!=2 || (std::strcmp(argv[1],"--demo") && std::strcmp(argv[1],"--check") && std::strcmp(argv[1],"--benchmark"))) {
        std::fprintf(stderr,"Usage: %s --demo | --check | --benchmark\n",argv[0]);return EXIT_FAILURE;
    }
    CUDA_CHECK(cudaSetDevice(0));
    cudaDeviceProp prop{};
    CUDA_CHECK(cudaGetDeviceProperties(&prop,0));
    std::printf("GPU: %s; SMs: %d\n",prop.name,prop.multiProcessorCount);
    if(!std::strcmp(argv[1],"--demo")) {
        Data data(10);
        auto direct=data.validate(0,4),shared=data.validate(1,4);
        std::puts("N=10, B=4: three blocks, last block has two valid outputs.");
        std::puts(" i     a[i]   direct   shared  CPU reference");
        for(int i=0;i<10;++i) std::printf("%2d %8.2f %8.2f %8.2f %8.2f\n",i,data.a[i],direct[i],shared[i],
                                           i==0?0.f:data.a[i]-data.a[i-1]);
        std::puts("Both GPU outputs validated, including b[0]=0 and the partial final block.");
        return EXIT_SUCCESS;
    }
    if(!std::strcmp(argv[1],"--check")) {
        int checks=0;
        for(int n:{1,2,3,4,5,31,32,33,255,256,257,1003}) {
            Data data(n);
            for(int threads:{1,4,32,64,128,256}) {
                data.validate(0,threads);data.validate(1,threads);++checks;
            }
        }
        std::printf("All %d length/block-size combinations passed for both kernels.\n",checks);
        return EXIT_SUCCESS;
    }
    int driver=0,runtime=0;
    CUDA_CHECK(cudaDriverGetVersion(&driver));CUDA_CHECK(cudaRuntimeGetVersion(&runtime));
    std::printf("L2 cache: %d bytes; driver: %d; runtime: %d; toolkit: %d\n",
                prop.l2CacheSize,driver,runtime,CUDART_VERSION);
    std::puts("7 trials per method; seed: 20261006; target batch: 5 ms; maximum repetitions: 200.");
    std::mt19937 rng(20261006);
    for(int n:{4099,65539,1048579,8388611})
        for(int threads:{128,256}) benchmark(n,threads,rng);
    std::puts("\nTimes exclude allocation, transfers, validation and printing.");
    std::puts("Repeated arrays can be cached. Fewer source-level loads need not mean less DRAM traffic.");
}


## 6. Compile

The timer implementation is compiled together with the host program. The kernel
definitions are included from the header. Compilation stops on an error.


In [ ]:
%%bash
set -eu
rm -f adjacent_shared_demo
nvcc -O2 -std=c++14 -Icode code/chapter02_adjacent_shared.cu code/TimingGPU.cu -o adjacent_shared_demo


## 7. Print actual GPU results

The program uses N=10 and B=4, matching the partial-block figure. The input is
`a[i]=0.25*(i%17)^2`. Here the first values are 0, 0.25, 1, 2.25, ... .
The output table contains the direct GPU result, the shared GPU result and the CPU
reference. Both GPU result columns are copied from device memory. No CPU-only
fallback is used if a GPU is unavailable.


In [ ]:
import subprocess

def run_cuda(mode):
    """Show program output in this cell, without writing result files."""
    result = subprocess.run(['./adjacent_shared_demo', mode],
                            capture_output=True, text=True)
    print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='')
    result.check_returncode()

run_cuda('--demo')


## 8. Check edge cases before timing

This runs 72 length/block-size combinations for both kernels, including N=1,
partial blocks and blocks much larger than the array. These tiny cases test
correctness; they do not measure useful GPU throughput.


In [ ]:
run_cuda('--check')


## 9. Compare direct and shared-memory timing

The benchmark uses four vector lengths and blocks of 128 or 256 threads.
Every configuration is validated. After warm-up, a pilot selects 1–200 repetitions
per batch, targeting about 5 ms (the cap can produce a shorter batch). Seven trials
are recorded for each method; their order is randomized. The output reports the
median and quartiles of the mean time per launch in each batch.

Allocation, input construction, transfers, validation and printing are outside
the timed intervals. Arrays are reused and may be cached; host submission gaps
may enter very short event intervals. The quartiles describe variability, not
statistical confidence bounds.

**Read the ratio:** T_direct/T_shared > 1 means shared is faster; < 1 means direct
is faster. Do not assume that shared must win. For an interior block the global
input-read count decreases from 2B to B+1, but the shared kernel also performs
shared-memory stores, shared-memory reads and a `__syncthreads()` barrier.
For the direct kernel, hardware caches can already supply neighboring threads
with recently fetched inputs, without these explicit copies or this block
barrier. Shared memory provides a programmer-managed buffer, whereas caching
is managed by hardware. With only two uses per input, the extra work may
outweigh the benefit of fewer source-level global reads. This is why adjacent differences are a low-reuse example.


In [ ]:
run_cuda('--benchmark')


## 10. Reuse the visual language for a wider stencil

The following is a **preview of the next example**, not an additional benchmark.
For a centered moving average of radius two, each output uses five inputs.
Four outputs require 20 direct input reads, but their combined input region has
only eight distinct values: four central values and two halo values on either side.

The drawing function `dependencies(B, left, right, j)` can be reused with
different stencil reaches. For adjacent differences, choose left=1 and right=0;
for a centered radius-R stencil, choose left=right=R. The central tile and halo
retain their colors. A wider halo requires its own cooperative loading scheme;
the current adjacent-difference kernel is not a moving-average implementation.


In [ ]:
fig = dependencies(B=4, left=2, right=2, j=8)
display(fig)
plt.close(fig)


The shared-memory allocation and barrier follow the
[CUDA programming guide](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/writing-cuda-kernels.html).
Timing support derives from the author's supplied Timing archive, with resource
management and error checks added as documented in the preceding notebooks.
